# DSPy Predictions and Examples

This notebook covers how to work with DSPy's data and prediction objects.

**What You'll Learn:**
- How to create and inspect `dspy.Prediction` objects
- How to create `dspy.Example` with `with_inputs()`
- How to write metric functions
- How to evaluate modules with `dspy.Evaluate`
- How to work with datasets from `utils.datasets`

## Setup

In [ ]:
import os
import sys
sys.path.append('../../')

import dspy
from utils import setup_default_lm, print_step, print_result, print_error, configure_dspy
from utils.datasets import get_sample_qa_data, get_sample_classification_data
from dotenv import load_dotenv

load_dotenv('../../.env')

try:
    lm = setup_default_lm(provider="openai", model="gpt-4o", max_tokens=500)
    configure_dspy(lm=lm)
    print_result("Language model configured successfully!", "Status")
except Exception as e:
    print_error(f"Failed to configure language model: {e}")
    print("Make sure you have set your OPENAI_API_KEY in the .env file")

## Example 1: Prediction Objects

Predictions hold model outputs. Access fields with dot notation.

In [ ]:
class QA(dspy.Signature):
    """Answer the question concisely."""
    question = dspy.InputField()
    answer = dspy.OutputField()

qa = dspy.Predict(QA)
prediction = qa(question="What is the largest ocean on Earth?")

# Accessing fields
print(f"prediction.answer = {prediction.answer}")
print(f"type(prediction) = {type(prediction).__name__}")

# Creating predictions manually
manual_pred = dspy.Prediction(answer="The Pacific Ocean", confidence="high")
print(f"\nManual prediction: {manual_pred.answer}, confidence: {manual_pred.confidence}")

## Example 2: Example Objects

Examples represent labeled data. Use `with_inputs()` to mark which fields are inputs vs labels.

In [ ]:
# Create examples with with_inputs()
examples = [
    dspy.Example(question="What is 2+2?", answer="4").with_inputs("question"),
    dspy.Example(question="Capital of France?", answer="Paris").with_inputs("question"),
    dspy.Example(question="Largest planet?", answer="Jupiter").with_inputs("question"),
]

print("Created examples with with_inputs():")
for i, ex in enumerate(examples, 1):
    print(f"  {i}. Q: {ex.question} -> A: {ex.answer}")

# with_inputs() marks which fields are inputs vs labels
ex = examples[0]
print(f"\nex.inputs() = {ex.inputs()}")
print(f"ex.labels() = {ex.labels()}")

## Example 3: Loading Datasets

The `utils.datasets` module provides sample datasets for learning.

In [ ]:
qa_data = get_sample_qa_data()
print(f"QA dataset: {len(qa_data)} examples")
for ex in qa_data[:3]:
    print(f"  Q: {ex.question[:50]}... -> A: {ex.answer[:30]}...")

class_data = get_sample_classification_data()
print(f"\nClassification dataset: {len(class_data)} examples")
for ex in class_data[:3]:
    print(f"  Text: {ex.text[:40]}... -> Label: {ex.label}")

## Example 4: Metric Functions

Metrics evaluate predictions against ground truth. They take `(example, prediction, trace)` and return a score.

In [ ]:
def exact_match(example, prediction, trace=None):
    """Strict exact match metric."""
    return prediction.answer.strip().lower() == example.answer.strip().lower()

def fuzzy_match(example, prediction, trace=None):
    """Fuzzy match: check if key words from the answer appear in prediction."""
    pred_words = set(prediction.answer.lower().split())
    true_words = set(example.answer.lower().split())
    overlap = len(pred_words & true_words)
    if not true_words:
        return 0.0
    return overlap / len(true_words)

# Test metrics
test_example = dspy.Example(question="What is the capital of France?", answer="Paris").with_inputs("question")
test_pred = qa(question=test_example.question)

exact = exact_match(test_example, test_pred)
fuzzy = fuzzy_match(test_example, test_pred)

print(f"Question: {test_example.question}")
print(f"Expected: {test_example.answer}")
print(f"Predicted: {test_pred.answer}")
print(f"Exact match: {exact}")
print(f"Fuzzy match: {fuzzy:.2f}")

## Example 5: Module Evaluation

Evaluate a module's performance across a dataset.

In [ ]:
eval_set = [
    dspy.Example(question="What is 2+2?", answer="4").with_inputs("question"),
    dspy.Example(question="Capital of Japan?", answer="Tokyo").with_inputs("question"),
    dspy.Example(question="Largest planet?", answer="Jupiter").with_inputs("question"),
]

# Manual evaluation loop
print("Manual evaluation:")
scores = []
for ex in eval_set:
    pred = qa(question=ex.question)
    score = fuzzy_match(ex, pred)
    scores.append(score)
    print(f"  Q: {ex.question} | Expected: {ex.answer} | Got: {pred.answer} | Score: {score:.2f}")

avg_score = sum(scores) / len(scores)
print(f"\nAverage score: {avg_score:.2f}")

# Using dspy.Evaluate
print("\nUsing dspy.Evaluate:")
try:
    evaluator = dspy.Evaluate(
        devset=eval_set,
        metric=fuzzy_match,
        num_threads=1,
        display_progress=True
    )
    overall_score = evaluator(qa)
    print(f"Overall evaluation score: {overall_score}")
except Exception as e:
    print(f"dspy.Evaluate encountered an error: {e}")
    print("(Manual evaluation above works as a fallback)")

## Summary

**Key Takeaways:**
1. Predictions hold model outputs — access fields with dot notation
2. Examples represent labeled data — use `with_inputs()` to mark input fields
3. `ex.inputs()` returns input fields, `ex.labels()` returns label fields
4. Metrics take `(example, prediction, trace)` and return a score
5. `dspy.Evaluate` runs systematic evaluation across a dataset